# CELL 01

# Knowledge Graph — Notebook 7
## RDF and SPARQL for Travel Planning

### From Graph Representation to Semantic Web Querying

Running Problem:

> "I want to plan a trip using structured travel knowledge."

In the previous notebooks, we represented travel knowledge using:

    Python Graph
          ↓
    Relational Database
          ↓
    Vector Store
          ↓
    Neo4j + Cypher

In this notebook, we will represent the same knowledge using:

    RDF + SPARQL

# CELL 02

## We Continue the Same Travel Problem

We will NOT introduce a new domain.

We continue with the travel knowledge developed
in the previous notebooks.

For example:

    Chennai → located in → Tamil Nadu

    Mahabalipuram → near → Chennai

    Mahabalipuram → has category → Heritage

    Hotel SeaView → located in → Mahabalipuram

    Hotel SeaView → price per night → ₹3500

The purpose is to learn a new representation
of the SAME knowledge.

# CELL 03

## Recall: Knowledge as a Triple

We have already seen:

    Subject → Relationship → Object

For example:

    Mahabalipuram → NEAR → Chennai

This is a triple.

RDF is based fundamentally on this
triple representation.

# CELL 04

## What is RDF?

RDF stands for:

> Resource Description Framework

RDF is a standard model for representing
information about resources.

At its core:

    RDF = Subject + Predicate + Object

For example:

    Mahabalipuram
          |
        NEAR
          |
       Chennai

The important point:

> RDF provides a standard way of representing
> relationships between resources.

#Why do we need a standard
# CELL 05

## Why a Standard Representation?

Suppose different systems represent the same fact differently:

System A:

    Mahabalipuram, NEAR, Chennai

System B:

    Mahabalipuram -> close_to -> Chennai

System C:

    Chennai <- nearby <- Mahabalipuram

The meaning may be similar, but machines need
a consistent representation.

RDF provides a standardized model for expressing
such information.

# CELL 06

## RDF Triple

An RDF statement has three components:

    SUBJECT
       ↓
    PREDICATE
       ↓
    OBJECT

Example:

    Mahabalipuram
          |
         NEAR
          |
       Chennai

Therefore:

Subject   = Mahabalipuram
Predicate = NEAR
Object    = Chennai

# CELL 07

## Entity versus Literal

Consider:

    Mahabalipuram → NEAR → Chennai

Here:

    Mahabalipuram = resource
    Chennai       = resource

Now consider:

    Hotel SeaView → PRICE_PER_NIGHT → 3500

Here:

    Hotel SeaView = resource
    3500          = literal value

This distinction becomes important in RDF.

# STEP 2
# CELL 08

## Implementing RDF

We will use Python with RDFLib.

RDFLib allows us to:

    create RDF graphs
    add RDF triples
    inspect RDF triples
    execute SPARQL queries

Our objective is not to memorize RDFLib syntax.

The objective is to understand:

    Knowledge
       ↓
    RDF representation
       ↓
    SPARQL query
       ↓
    Answer

In [1]:
#Install RDFLIB
# CELL 09

%pip install rdflib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# CELL 10

from rdflib import Graph, Namespace, URIRef, Literal

# CELL 11

## Create an RDF Graph

An RDF graph is a collection of RDF triples.

We will create an empty graph and gradually
add our travel knowledge.

In [3]:
# CELL 11

from rdflib import Graph

travel_graph = Graph()

print("Empty RDF graph created.")

Empty RDF graph created.


## STEP 3 Namespaces and URI
# CELL 12

## Why do RDF resources need identifiers?

In ordinary Python we can write:

    "Chennai"

But in a large distributed knowledge system,
we need identifiers that can uniquely identify resources.

RDF commonly uses URIs.

For teaching purposes, we will create our own
travel namespace.

In [4]:
# CELL 13

from rdflib import Namespace

TRAVEL = Namespace("http://example.org/travel/")

print(TRAVEL)

http://example.org/travel/


# creating resources
# CELL 14

## Representing Travel Resources

We can now represent:

    Chennai
    Bengaluru
    Mysuru
    Mahabalipuram
    Pondicherry
    Hotel SeaView
    Hotel Heritage

as RDF resources.

In [5]:
# CELL 15

chennai = TRAVEL.Chennai
bengaluru = TRAVEL.Bengaluru
mysuru = TRAVEL.Mysuru

mahabalipuram = TRAVEL.Mahabalipuram
pondicherry = TRAVEL.Pondicherry

hotel_seaview = TRAVEL.HotelSeaView
hotel_heritage = TRAVEL.HotelHeritage

tamil_nadu = TRAVEL.TamilNadu
heritage = TRAVEL.Heritage

## STEP 4 -PREDICATES
# CELL 16

## Representing Relationships

We also need identifiers for relationships.

For example:

    LOCATED_IN
    NEAR
    CONNECTED_TO
    HAS_CATEGORY
    PRICE_PER_NIGHT

We represent these as RDF predicates.

In [6]:
# CELL 17

LOCATED_IN = TRAVEL.locatedIn
NEAR = TRAVEL.near
CONNECTED_TO = TRAVEL.connectedTo
HAS_CATEGORY = TRAVEL.hasCategory
PRICE_PER_NIGHT = TRAVEL.pricePerNight

## FIRST RDF TRIPLE
# CELL 18

## Create Our First RDF Triple

We want to represent:

    Chennai → LOCATED_IN → Tamil Nadu

In RDFLib:

    graph.add((subject, predicate, object))

In [7]:
# CELL 19

travel_graph.add(
    (chennai, LOCATED_IN, tamil_nadu)
)

print("Triple added.")

Triple added.


In [8]:
#Inspect
# CELL 20

for triple in travel_graph:
    print(triple)

(rdflib.term.URIRef('http://example.org/travel/Chennai'), rdflib.term.URIRef('http://example.org/travel/locatedIn'), rdflib.term.URIRef('http://example.org/travel/TamilNadu'))


# CELL 21

## Observe the RDF Triple

The output may look less readable than:

    Chennai → located in → Tamil Nadu

because RDFLib internally represents resources
using URIs.

The important idea is still:

    Subject
       ↓
    Predicate
       ↓
    Object

RDF has not changed the fundamental idea of
the knowledge graph.

It has standardized its representation.

## Build the RDF graph
# CELL 22

## Build the Travel Knowledge Graph

Now we will add the remaining travel facts.

We will use the same knowledge used in
the earlier notebooks.

This is important:

> We are changing the representation,
> not the knowledge.

In [9]:
# CELL 23

travel_graph.add((mahabalipuram, NEAR, chennai))
travel_graph.add((mahabalipuram, HAS_CATEGORY, heritage))

travel_graph.add((pondicherry, NEAR, chennai))
travel_graph.add((pondicherry, HAS_CATEGORY, heritage))

travel_graph.add((hotel_seaview, LOCATED_IN, mahabalipuram))
travel_graph.add((hotel_heritage, LOCATED_IN, pondicherry))

travel_graph.add((chennai, CONNECTED_TO, bengaluru))
travel_graph.add((bengaluru, CONNECTED_TO, mysuru))

<Graph identifier=N9f1783c8a32f4df5b2bc958106d1d118 (<class 'rdflib.graph.Graph'>)>

#Price Literals
# CELL 24

## Adding Literal Values

A price is not another travel entity.

It is a literal value.

Therefore:

    Hotel SeaView → pricePerNight → 3500

will use an RDF Literal for 3500.

In [10]:
# CELL 25

travel_graph.add(
    (hotel_seaview, PRICE_PER_NIGHT, Literal(3500))
)

travel_graph.add(
    (hotel_heritage, PRICE_PER_NIGHT, Literal(3000))
)

<Graph identifier=N9f1783c8a32f4df5b2bc958106d1d118 (<class 'rdflib.graph.Graph'>)>

In [11]:
#Count Triples
# CELL 26

print("Number of RDF triples:", len(travel_graph))

Number of RDF triples: 11


In [12]:
#Display all Triples
# CELL 27

for subject, predicate, object_ in travel_graph:
    print(subject, predicate, object_)

http://example.org/travel/Chennai http://example.org/travel/connectedTo http://example.org/travel/Bengaluru
http://example.org/travel/Mahabalipuram http://example.org/travel/near http://example.org/travel/Chennai
http://example.org/travel/Pondicherry http://example.org/travel/hasCategory http://example.org/travel/Heritage
http://example.org/travel/HotelSeaView http://example.org/travel/pricePerNight 3500
http://example.org/travel/Mahabalipuram http://example.org/travel/hasCategory http://example.org/travel/Heritage
http://example.org/travel/Chennai http://example.org/travel/locatedIn http://example.org/travel/TamilNadu
http://example.org/travel/Pondicherry http://example.org/travel/near http://example.org/travel/Chennai
http://example.org/travel/HotelHeritage http://example.org/travel/locatedIn http://example.org/travel/Pondicherry
http://example.org/travel/HotelHeritage http://example.org/travel/pricePerNight 3000
http://example.org/travel/HotelSeaView http://example.org/travel/locate

#Step 7 — RDF Graph visualization concept
# CELL 28

## RDF is Still a Graph

Our RDF representation can still be visualized as:

    Mahabalipuram ──NEAR────────> Chennai
          |
          |
     HAS_CATEGORY
          |
          ↓
       Heritage

The difference is not the graph idea.

The difference is that RDF gives us
a standardized representation using
resources, predicates and literals.

#Step 8 — SPARQL
# CELL 29

## Now We Need a Query Language

We have built the RDF graph.

How do we ask questions?

For RDF graphs, we can use:

> SPARQL

SPARQL is a query language for RDF data.

We can think of the relationship as:

    RDF
     ↓
    stores knowledge

    SPARQL
     ↓
    queries knowledge

#Cypher vs SPARQL
# CELL 30

## We Have Already Seen Another Graph Query Language

In KG-06 we used:

    Neo4j → Cypher

Now we use:

    RDF → SPARQL

Both allow us to ask questions about
relationships in graph data.

But their syntax and underlying data models
are different.

## Step 9 — First SPARQL Query
# CELL 31

## Question 1

Where is Chennai located?

Knowledge:

    Chennai → locatedIn → Tamil Nadu

We want to retrieve the object.

In SPARQL we describe the pattern:

    Chennai → locatedIn → ?

The question mark represents a variable.

In [13]:
# CELL 32

query = """
SELECT ?location
WHERE {
    <http://example.org/travel/Chennai>
        <http://example.org/travel/locatedIn>
        ?location .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.location)

http://example.org/travel/TamilNadu


# CELL 33

## Understanding the SPARQL Pattern

The query says:

    Find something that is:

    Chennai
       ↓
    locatedIn
       ↓
       ?

The ?location variable captures the unknown object.

This is very similar to asking:

> "What is connected to Chennai through
> the LOCATED_IN relationship?"

## Step 10 — Query places near Chennai
# CELL 34

## Question 2

Which places are near Chennai?

We know:

    Mahabalipuram → near → Chennai

    Pondicherry → near → Chennai

We want:

    ?place → near → Chennai

In [14]:
# CELL 35

query = """
SELECT ?place
WHERE {
    ?place
        <http://example.org/travel/near>
        <http://example.org/travel/Chennai> .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.place)

http://example.org/travel/Mahabalipuram
http://example.org/travel/Pondicherry


#Step 11 — Use namespace in SPARQL
# CELL 36

## Making SPARQL Easier to Read

Writing complete URIs repeatedly is inconvenient.

Instead we can define a prefix:

    PREFIX travel:
        <http://example.org/travel/>

Then we can write:

    travel:Chennai

instead of:

    <http://example.org/travel/Chennai>

In [15]:
# CELL 37

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?place
WHERE {
    ?place travel:near travel:Chennai .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.place)

http://example.org/travel/Mahabalipuram
http://example.org/travel/Pondicherry


## Step 12 — Heritage destinations
# CELL 38

## Question 3

Which destinations are heritage destinations?

Knowledge pattern:

    ?destination
          |
     hasCategory
          |
       Heritage

SPARQL allows us to describe this graph pattern
directly.

In [16]:
# CELL 39

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?destination
WHERE {
    ?destination travel:hasCategory travel:Heritage .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.destination)

http://example.org/travel/Mahabalipuram
http://example.org/travel/Pondicherry


##Step 13 — Relationship chaining
# CELL 40

## Question 4

Which heritage destinations are near Chennai?

We need TWO conditions:

    ?destination → hasCategory → Heritage

AND

    ?destination → near → Chennai

This is a graph pattern involving two relationships.

In [17]:
# CELL 41

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?destination
WHERE {
    ?destination travel:hasCategory travel:Heritage .
    ?destination travel:near travel:Chennai .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.destination)

http://example.org/travel/Mahabalipuram
http://example.org/travel/Pondicherry


##Step 14 — Hotels
# CELL 42

## Question 5

Which hotels are located in Mahabalipuram?

Pattern:

    ?hotel
       |
    locatedIn
       |
    Mahabalipuram

In [18]:
# CELL 43

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?hotel
WHERE {
    ?hotel travel:locatedIn travel:Mahabalipuram .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.hotel)

http://example.org/travel/HotelSeaView


##Step 15 — Retrieve hotel and price
# CELL 44

## Question 6

Show:

    Hotel name
    Price

We need two pieces of information about the same hotel.

Pattern:

    ?hotel → locatedIn → ?destination

    ?hotel → pricePerNight → ?price

In [19]:
# CELL 45

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?hotel ?price
WHERE {
    ?hotel travel:pricePerNight ?price .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.hotel, row.price)

http://example.org/travel/HotelSeaView 3500
http://example.org/travel/HotelHeritage 3000


##Step 16 — Filtering

# CELL 46

## Question 7

Which hotels cost less than ₹3500?

This is different from simple graph traversal.

We now combine:

    graph pattern
          +
    numerical condition

SPARQL provides:

    FILTER

In [20]:
# CELL 47

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?hotel ?price
WHERE {
    ?hotel travel:pricePerNight ?price .
    FILTER(?price < 3500)
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.hotel, row.price)

http://example.org/travel/HotelHeritage 3000


##Step 17 — Multi-hop reasoning
# CELL 48

## Question 8

Can we travel from Chennai to Mysuru
through connected cities?

Our knowledge contains:

    Chennai
       |
    CONNECTED_TO
       ↓
    Bengaluru
       |
    CONNECTED_TO
       ↓
    Mysuru

This is a multi-hop relationship.

# CELL 49

## SPARQL Property Paths

SPARQL supports property paths.

For example:

    travel:connectedTo+

means:

> one or more connectedTo relationships.

This allows us to express multi-hop connectivity.

In [21]:
# CELL 50

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?city
WHERE {
    travel:Chennai travel:connectedTo+ ?city .
}
"""

results = travel_graph.query(query)

for row in results:
    print(row.city)

http://example.org/travel/Bengaluru
http://example.org/travel/Mysuru


## Step 18 — Multi-hop reasoning chain
# CELL 51

## Observe the Reasoning Chain

The graph contains:

    Chennai
       ↓
    CONNECTED_TO
       ↓
    Bengaluru
       ↓
    CONNECTED_TO
       ↓
    Mysuru

The query can traverse the graph through
multiple relationships.

This is the same fundamental idea we explored
with BFS in our Python notebooks and
property paths in Neo4j.

## Step 19 — Combined travel query
# CELL 52

## Question 9 — A Real Travel Question

Find:

> Heritage destinations near Chennai
> and their hotels with price below ₹3500.

We need to combine:

    Destination
         ↓
      NEAR
         ↓
      Chennai

and

    Destination
         ↓
    HAS_CATEGORY
         ↓
      Heritage

and

    Hotel
         ↓
    LOCATED_IN
         ↓
    Destination

and

    Hotel
         ↓
    PRICE_PER_NIGHT
         ↓
      < 3500

In [22]:
# CELL 53

query = """
PREFIX travel: <http://example.org/travel/>

SELECT ?destination ?hotel ?price
WHERE {
    ?destination travel:near travel:Chennai .
    ?destination travel:hasCategory travel:Heritage .
    
    ?hotel travel:locatedIn ?destination .
    ?hotel travel:pricePerNight ?price .
    
    FILTER(?price < 3500)
}
"""

results = travel_graph.query(query)

for row in results:
    print(
        "Destination:", row.destination,
        "| Hotel:", row.hotel,
        "| Price:", row.price
    )

Destination: http://example.org/travel/Pondicherry | Hotel: http://example.org/travel/HotelHeritage | Price: 3000


## Step 20 — Question → Graph pattern → SPARQL
# CELL 54

## The Querying Process

We should not begin by writing SPARQL syntax.

Start with the question:

    "Find affordable hotels in heritage
     destinations near Chennai."

Then identify:

### Entities

    Chennai
    Destination
    Heritage
    Hotel

### Relationships

    near
    hasCategory
    locatedIn
    pricePerNight

### Graph pattern

    Destination → near → Chennai

    Destination → hasCategory → Heritage

    Hotel → locatedIn → Destination

    Hotel → pricePerNight → Price

### Condition

    Price < 3500

Only then write the SPARQL query.

This is the same reasoning process we used
with Cypher.

## Step 21 — RDF versus Neo4j
# CELL 55

## RDF + SPARQL versus Neo4j + Cypher

We have now represented essentially the same
travel knowledge in two graph approaches.

### Neo4j

    Graph database
    Property graph model
    Cypher query language

### RDF

    Standard graph-based data model
    Subject–Predicate–Object triples
    SPARQL query language

Both support relationship-oriented querying.

But they come from different ecosystems
and have different design goals.

# CELL 56

## Same Question — Different Query Languages

Question:

> Which destinations are near Chennai?

### Cypher

Conceptually:

    (destination)-[:NEAR]->(Chennai)

### SPARQL

Conceptually:

    ?destination travel:near travel:Chennai

The question is the same.

The representation and query language differ.

## Step 22 — Important distinction
# CELL 57

## An Important Conceptual Distinction

Do not conclude:

    "Neo4j is better than RDF"

or:

    "RDF is better than Neo4j."

That is not the lesson.

The better question is:

> What representation and technology best
> suits the problem?

Neo4j is a powerful graph database.

RDF is a standardized model for representing
and linking knowledge.

SPARQL provides a standard query language
for RDF graphs.

## Step 23 — Why RDF matters
# CELL 58

## Why is RDF Important?

RDF becomes particularly important when
knowledge needs to be:

- represented using standards
- shared across systems
- linked across datasets
- interpreted consistently
- queried using standard semantic-web technologies

This leads to an important idea:

> A Knowledge Graph is not only about storing
> connections; it can also be about making
> knowledge interoperable.

## Step 24 — RDF and the Semantic Web
# CELL 59

## RDF and the Semantic Web

RDF is one of the foundational technologies
associated with the Semantic Web.

The broad idea is:

    Data
      ↓
    Meaning
      ↓
    Relationships
      ↓
    Machine-processable knowledge
      ↓
    Linked knowledge

This is one reason RDF is widely discussed
in Knowledge Graph research.

## Step 25 — RDF graph serialization
# CELL 60

## RDF Can Also Be Serialized

RDF is a model.

It can be written in different concrete formats.

Examples include:

    Turtle
    RDF/XML
    JSON-LD

We will briefly look at Turtle because
it is particularly readable for humans.

# CELL 61

print(
    travel_graph.serialize(
        format="turtle"
    )
)

# CELL 62

## Observe the Turtle Representation

The same knowledge that we created using Python
can be serialized in a compact RDF syntax.

For example, conceptually:

    travel:Mahabalipuram
        travel:near travel:Chennai ;
        travel:hasCategory travel:Heritage .

The underlying knowledge has not changed.

Only the representation has changed.

This is a very important Knowledge Graph concept:

> Knowledge representation and knowledge storage
> are related, but they are not the same thing.

## Step 26 — Serialization and interchange
# CELL 63

## Why Serialization Matters

Suppose one system creates RDF knowledge
and another system needs to consume it.

A standard serialization allows the graph
to be exchanged between systems.

Conceptually:

    System A
       ↓
    RDF
       ↓
    Turtle / JSON-LD / RDF/XML
       ↓
    System B

This supports knowledge interoperability.

# CELL 64

## What Have We Learned?

We started with:

    Travel knowledge

We represented it as:

    RDF triples

We stored it in:

    an RDF graph

We queried it using:

    SPARQL

We performed:

    relationship queries
    filtering
    multi-hop traversal
    combined graph patterns

# CELL 65

## KG-07 — Representation Journey

Our Knowledge Graph journey is now:

    Real-world travel knowledge
             ↓
          Triples
             ↓
       Python Graph
             ↓
      Relational DB
             ↓
       Vector Store
             ↓
     Neo4j + Cypher
             ↓
       RDF + SPARQL

The knowledge remained essentially the same.

The representation changed.

# CELL 66

#  Summary

## RDF

    Subject → Predicate → Object

## RDF Graph

    Collection of RDF triples

## SPARQL

    Query language for RDF graphs

## Neo4j

    Graph database

## Cypher

    Query language for Neo4j

## Central idea

> Same knowledge can be represented
> in different ways depending on
> the purpose and ecosystem.

# CELL 67

## Exercises

### Exercise 1

Add:

    Hyderabad → CONNECTED_TO → Chennai

Query all cities connected to Chennai.

---

### Exercise 2

Add:

    Hyderabad → HAS_CATEGORY → Heritage

Query all heritage destinations.

---

### Exercise 3

Add:

    Charminar Hotel → LOCATED_IN → Hyderabad

    Charminar Hotel → PRICE_PER_NIGHT → 2800

Find hotels costing less than ₹3500.

---

### Exercise 4

Write a SPARQL query to find:

    Heritage destinations near Chennai.

---

### Exercise 5

Write a SPARQL query to find:

    Hotels located in heritage destinations.

---

### Exercise 6

Write a multi-hop SPARQL query to find
cities reachable from Chennai.

# CELL 68

## Reflection Questions

1. What is the difference between an RDF resource
   and an RDF literal?

2. Why does RDF use subject–predicate–object?

3. What problem does SPARQL solve?

4. What is a URI?

5. How is SPARQL different from Cypher?

6. What is a property path?

7. Why can RDF support knowledge interoperability?

8. Is RDF "better" than Neo4j?

9. When would you prefer Neo4j?

10. When might RDF be more appropriate?

# CELL 69

# Final Takeaway

> **RDF gives us a standard way to represent
> knowledge as interconnected triples,
> while SPARQL allows us to query that knowledge
> through graph patterns.**

The important progression is:

    Knowledge
       ↓
    Representation
       ↓
    Graph
       ↓
    Query
       ↓
    Reasoning

We have now learned two important graph ecosystems:

    Neo4j + Cypher

and

    RDF + SPARQL